<a href="https://colab.research.google.com/github/manjunath-t-g/NLP-Sentiment-Analysis/blob/main/Ai%26Ml_prj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio textblob pandas


In [ ]:
import gradio as gr
from textblob import TextBlob
import re
from collections import Counter
import pandas as pd


# -----------------------------
# Text preprocessing
# -----------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# -----------------------------
# Sentiment analysis
# -----------------------------
def analyze_sentiment(text):

    if not text.strip():
        return "Please enter some text.", "", "", ""

    blob = TextBlob(text)

    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    if polarity > 0.1:
        sentiment = "😊 Positive"
    elif polarity < -0.1:
        sentiment = "😞 Negative"
    else:
        sentiment = "😐 Neutral"

    cleaned = clean_text(text)

    words = cleaned.split()

    stopwords = {
        "the", "is", "a", "an", "and", "or", "to",
        "of", "in", "on", "for", "with", "this",
        "that", "it", "was", "are", "i", "you"
    }

    filtered_words = [
        word for word in words
        if word not in stopwords
    ]

    frequency = Counter(filtered_words)

    # Top 10 words
    top_words = frequency.most_common(10)

    if top_words:
        frequency_text = "\n".join(
            [f"{word}: {count}" for word, count in top_words]
        )
    else:
        frequency_text = "No words found."

    # Statistics
    statistics = f"""
Characters: {len(text)}
Words: {len(words)}
Unique Words: {len(set(words))}
Polarity: {polarity:.3f}
Subjectivity: {subjectivity:.3f}
"""

    return (
        sentiment,
        cleaned,
        statistics,
        frequency_text
    )


# -----------------------------
# Create UI
# -----------------------------
with gr.Blocks(title="NLP Sentiment Analyzer") as app:

    gr.Markdown(
        """
        # 🧠 NLP Sentiment Analysis Dashboard

        Enter any text below and the application will perform
        basic NLP preprocessing and sentiment analysis.
        """
    )

    text_input = gr.Textbox(
        label="Enter Text",
        placeholder="Example: I really enjoyed this product. It is amazing!",
        lines=6
    )

    analyze_button = gr.Button(
        "🔍 Analyze Text",
        variant="primary"
    )

    gr.Markdown("## 📊 Results")

    sentiment_output = gr.Textbox(
        label="Sentiment"
    )

    cleaned_output = gr.Textbox(
        label="Cleaned Text",
        lines=3
    )

    statistics_output = gr.Textbox(
        label="Text Statistics",
        lines=6
    )

    frequency_output = gr.Textbox(
        label="Most Frequent Words",
        lines=10
    )

    analyze_button.click(
        fn=analyze_sentiment,
        inputs=text_input,
        outputs=[
            sentiment_output,
            cleaned_output,
            statistics_output,
            frequency_output
        ]
    )


# Launch the application
app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a2f368e8b2f15f8e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# INSTALLATION
# ============================================================

# Run this once in Google Colab/Jupyter if needed:
# !pip install -q gradio textblob nltk


# ============================================================
# IMPORTS
# ============================================================

import gradio as gr
from textblob import TextBlob
import nltk
import re
from collections import Counter

from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag


# ============================================================
# DOWNLOAD NLTK DATA
# ============================================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")


# ============================================================
# NLP TOOLS
# ============================================================

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

STOPWORDS = set(stopwords.words("english"))


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):
    """
    Convert text to lowercase, remove URLs,
    punctuation and extra spaces.
    """

    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Keep only letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# CONVERT POS TAG TO WORDNET FORMAT
# ============================================================

def get_wordnet_pos(tag):
    """
    Convert NLTK POS tags to WordNet POS tags.
    This improves lemmatization.
    """

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    return wordnet.NOUN


# ============================================================
# NLP PREPROCESSING
# ============================================================

def preprocess_text(text):

    cleaned = clean_text(text)

    # Tokenization
    tokens = word_tokenize(cleaned)

    # Remove stopwords
    filtered_tokens = [
        word for word in tokens
        if word not in STOPWORDS
    ]

    # -------------------------
    # Stemming
    # -------------------------

    stemmed_words = [
        stemmer.stem(word)
        for word in filtered_tokens
    ]

    # -------------------------
    # Lemmatization
    # -------------------------

    tagged_words = pos_tag(filtered_tokens)

    lemmatized_words = [
        lemmatizer.lemmatize(
            word,
            get_wordnet_pos(tag)
        )
        for word, tag in tagged_words
    ]

    return (
        cleaned,
        filtered_tokens,
        stemmed_words,
        lemmatized_words
    )


# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

def analyze_sentiment(text):

    if not text or not text.strip():
        return (
            "⚠️ Please enter some text.",
            "",
            "",
            "",
            "",
            ""
        )

    # --------------------------------------------------------
    # Sentiment Analysis
    # --------------------------------------------------------

    blob = TextBlob(text)

    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    if polarity > 0.1:
        sentiment = "😊 Positive"

    elif polarity < -0.1:
        sentiment = "😞 Negative"

    else:
        sentiment = "😐 Neutral"

    # --------------------------------------------------------
    # NLP preprocessing
    # --------------------------------------------------------

    (
        cleaned,
        filtered_tokens,
        stemmed_words,
        lemmatized_words
    ) = preprocess_text(text)

    # --------------------------------------------------------
    # Word frequency
    # --------------------------------------------------------

    frequency = Counter(lemmatized_words)

    top_words = frequency.most_common(10)

    if top_words:

        frequency_text = "\n".join(
            f"{i}. {word} — {count}"
            for i, (word, count)
            in enumerate(top_words, start=1)
        )

    else:

        frequency_text = "No meaningful words found."

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    original_words = text.split()

    statistics = f"""
Characters: {len(text)}
Words: {len(original_words)}
Tokens after stopword removal: {len(filtered_tokens)}
Unique Words: {len(set(filtered_tokens))}
Sentences: {len(blob.sentences)}

Polarity: {polarity:.3f}
Subjectivity: {subjectivity:.3f}

Polarity:
-1.0 = Negative
 0.0 = Neutral
+1.0 = Positive

Subjectivity:
0.0 = Objective
1.0 = Subjective
"""

    # --------------------------------------------------------
    # Convert lists to readable text
    # --------------------------------------------------------

    token_text = " | ".join(filtered_tokens)

    stemmed_text = " | ".join(stemmed_words)

    lemmatized_text = " | ".join(lemmatized_words)

    return (
        sentiment,
        cleaned,
        token_text,
        stemmed_text,
        lemmatized_text,
        statistics + "\nTop Words:\n" + frequency_text
    )


# ============================================================
# CLEAR FUNCTION
# ============================================================

def clear_all():

    return "", "", "", "", "", ""


# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(
    title="NLP Sentiment Analyzer",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # 🧠 NLP Sentiment Analysis Dashboard

        Enter text to perform:

        **Cleaning → Tokenization → Stopword Removal →
        Stemming → Lemmatization → Sentiment Analysis**
        """
    )

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    text_input = gr.Textbox(
        label="📝 Enter Text",
        placeholder=(
            "Example: I really enjoyed this amazing product!"
        ),
        lines=6
    )

    with gr.Row():

        analyze_button = gr.Button(
            "🔍 Analyze Text",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    # --------------------------------------------------------
    # SENTIMENT
    # --------------------------------------------------------

    gr.Markdown("## 📊 Sentiment")

    sentiment_output = gr.Textbox(
        label="Sentiment",
        interactive=False
    )

    # --------------------------------------------------------
    # PREPROCESSING
    # --------------------------------------------------------

    gr.Markdown("## 🧹 NLP Preprocessing")

    cleaned_output = gr.Textbox(
        label="1️⃣ Cleaned Text",
        lines=3,
        interactive=False
    )

    token_output = gr.Textbox(
        label="2️⃣ Tokens / Stopwords Removed",
        lines=3,
        interactive=False
    )

    stemmed_output = gr.Textbox(
        label="3️⃣ Stemmed Words",
        lines=3,
        interactive=False
    )

    lemmatized_output = gr.Textbox(
        label="4️⃣ Lemmatized Words",
        lines=3,
        interactive=False
    )

    # --------------------------------------------------------
    # STATISTICS
    # --------------------------------------------------------

    gr.Markdown("## 📈 Statistics & Word Frequency")

    statistics_output = gr.Textbox(
        label="Analysis Statistics",
        lines=12,
        interactive=False
    )

    # --------------------------------------------------------
    # EXAMPLES
    # --------------------------------------------------------

    gr.Examples(
        examples=[
            [
                "I really loved this product. "
                "It was amazing and easy to use!"
            ],
            [
                "The service was terrible. "
                "I am extremely disappointed."
            ],
            [
                "The package arrived yesterday."
            ],
            [
                "The movie was running slowly, "
                "but the actors performed wonderfully."
            ]
        ],
        inputs=text_input
    )

    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    analyze_button.click(
        fn=analyze_sentiment,
        inputs=text_input,
        outputs=[
            sentiment_output,
            cleaned_output,
            token_output,
            stemmed_output,
            lemmatized_output,
            statistics_output
        ]
    )

    clear_button.click(
        fn=clear_all,
        inputs=[],
        outputs=[
            text_input,
            sentiment_output,
            cleaned_output,
            token_output,
            stemmed_output,
            lemmatized_output,
            statistics_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(share=True)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
/tmp/ipykernel_333/1170034202.py:269:

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0550fe80ddd87d6aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
